# 004 — MixIT Separation Quality Check

Inspect Bird MixIT separation outputs from exp 013 Phase 2.
Compare original soundscapes vs 4 separated channels using spectrograms and audio playback.

**Goal**: Validate separation quality before scaling to 10,592 unlabeled soundscapes.

**Data**: 66 labeled soundscape NPZ files in `shared/mixit/pretrained/soundscapes_labeled/`

**Questions**:
1. Does channel 0 contain the dominant species? Do other channels isolate background species?
2. How does separation quality differ for birds vs frogs vs insects?
3. Are there over-separation artifacts (single source split across channels)?

In [ ]:
import csv
from pathlib import Path

import IPython.display as ipd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf

MIXIT_SR = 22050
SEPARATED_DIR = Path(
    "/storage/project/ps-clef2026_birdclef-0/shared/mixit/pretrained/soundscapes_labeled"
)
SOUNDSCAPE_DIR = Path(
    "/storage/project/ps-clef2026_birdclef-0/shared/data/birdclef-2026/train_soundscapes"
)
LABELS_CSV = Path(
    "/storage/project/ps-clef2026_birdclef-0/shared/data/birdclef-2026/train_soundscapes_labels.csv"
)

# Load labels for reference
labels = {}
with open(LABELS_CSV) as f:
    reader = csv.DictReader(f)
    for row in reader:
        key = row["filename"]
        if key not in labels:
            labels[key] = []
        labels[key].append(
            {"start": row["start"], "end": row["end"], "species": row["primary_label"]}
        )

# Deduplicate
for key in labels:
    seen = set()
    deduped = []
    for seg in labels[key]:
        sig = (seg["start"], seg["end"], seg["species"])
        if sig not in seen:
            seen.add(sig)
            deduped.append(seg)
    labels[key] = deduped

print(f"Loaded labels for {len(labels)} files")
print(f"Separated NPZ files: {len(list(SEPARATED_DIR.glob('*.npz')))}")

In [ ]:
FIGURES_DIR = Path(
    "/storage/project/ps-clef2026_birdclef-0/mgustineli3/birdclef-2026/"
    "user/murilo/experiments/013-mixit-pseudo-labeling/figures"
)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def plot_separation(
    filename: str, start_sec: int = 0, duration_sec: int = 15, save_name: str | None = None
):
    """Plot spectrograms of original + 4 separated channels for a soundscape file.

    Args:
        filename: Soundscape filename (e.g., 'BC2026_Train_0039_S22_20211231_201500.ogg')
        start_sec: Start time in seconds for the visualization window
        duration_sec: Duration in seconds to show
        save_name: If set, save figure to FIGURES_DIR/{save_name}.jpg
    """
    # Load original audio
    audio_path = SOUNDSCAPE_DIR / filename
    original, orig_sr = sf.read(str(audio_path), dtype="float32")
    if original.ndim == 2:
        original = original.mean(axis=1)
    # Resample to MixIT rate for fair comparison
    if orig_sr != MIXIT_SR:
        original = librosa.resample(original, orig_sr=orig_sr, target_sr=MIXIT_SR)

    # Load separated channels
    npz_path = SEPARATED_DIR / f"{Path(filename).stem}.npz"
    data = np.load(npz_path)
    channels = data["channels"]  # (4, num_samples)

    # Trim to window
    start_sample = start_sec * MIXIT_SR
    end_sample = start_sample + duration_sec * MIXIT_SR
    original_clip = original[start_sample:end_sample]
    channel_clips = [channels[i, start_sample:end_sample] for i in range(4)]

    # Get labels for this file
    file_labels = labels.get(filename, [])
    species_in_window = set()
    for seg in file_labels:
        parts = seg["start"].split(":")
        seg_start = int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])
        parts = seg["end"].split(":")
        seg_end = int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])
        if seg_start < start_sec + duration_sec and seg_end > start_sec:
            for sp in seg["species"].split(";"):
                species_in_window.add(sp.strip())

    # Plot
    fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)

    all_clips = [original_clip] + channel_clips
    titles = ["Original"] + [f"Channel {i}" for i in range(4)]

    for ax, clip, title in zip(axes, all_clips, titles, strict=False):
        S = librosa.feature.melspectrogram(y=clip, sr=MIXIT_SR, n_mels=128, fmax=11025)
        S_db = librosa.power_to_db(S, ref=np.max)
        librosa.display.specshow(S_db, sr=MIXIT_SR, x_axis="time", y_axis="mel", ax=ax)
        rms = np.sqrt(np.mean(clip**2))
        ax.set_title(f"{title} (RMS: {rms:.4f})", fontsize=10)
        ax.set_ylabel("")

    species_str = ", ".join(sorted(species_in_window)) if species_in_window else "unknown"
    fig.suptitle(
        f"{filename}\n{start_sec}s–{start_sec + duration_sec}s | Species: {species_str}",
        fontsize=11,
    )
    plt.tight_layout()

    if save_name:
        fig_path = FIGURES_DIR / f"{save_name}.jpg"
        fig.savefig(fig_path, dpi=150, bbox_inches="tight", format="jpg")
        print(f"Saved: {fig_path}")

    plt.show()

    # Audio playback widgets
    print(f"\n{'=' * 60}")
    print(f"Audio playback: {filename} ({start_sec}s–{start_sec + duration_sec}s)")
    print(f"Species: {species_str}")
    print(f"{'=' * 60}")
    for title, clip in zip(titles, all_clips, strict=False):
        rms = np.sqrt(np.mean(clip**2))
        print(f"\n{title} (RMS: {rms:.4f}):")
        display(ipd.Audio(clip, rate=MIXIT_SR))

## Dense frog chorus (nocturnal, many overlapping species)

File 0039 from site S22 is a night recording with 4-5 co-occurring frog species — the densest multi-species scenario in the dataset.

In [ ]:
plot_separation(
    "BC2026_Train_0039_S22_20211231_201500.ogg",
    start_sec=0,
    duration_sec=15,
    save_name="dense_frog_chorus",
)

## Multi-species recording with bird activity

No dawn chorus files in the labeled set (all are evening/night, local 14:00–03:00).
Find a file with bird species (Aves) to test separation on birds, not just frogs.

In [ ]:
# Load taxonomy to identify bird species codes
import pandas as pd

taxonomy = pd.read_csv(
    "/storage/project/ps-clef2026_birdclef-0/shared/data/birdclef-2026/taxonomy.csv"
)
bird_codes = set(taxonomy[taxonomy["class_name"] == "Aves"]["primary_label"])

# Find files with the most bird species
bird_files = []
for fname in sorted(labels.keys()):
    all_species = {sp for seg in labels[fname] for sp in seg["species"].split(";")}
    bird_species = all_species & bird_codes
    if bird_species:
        bird_files.append((fname, len(bird_species), len(all_species)))

bird_files.sort(key=lambda x: -x[1])
print("Files with most bird species:")
for fname, n_birds, n_total in bird_files[:5]:
    print(f"  {fname}: {n_birds} birds / {n_total} total species")

In [ ]:
# Plot the file with most bird species (update filename after running cell above)
plot_separation(
    "BC2026_Train_0057_S15_20250617_060000.ogg",
    start_sec=0,
    duration_sec=15,
    save_name="bird_multi_species",
)

## Sparse/quiet recording (few species, low activity)

A recording with 1-2 species to see if MixIT avoids over-separation on simple inputs.

In [ ]:
# Find a sparse file (fewest species)
sparse_files = []
for fname in sorted(labels.keys()):
    species_set = {sp for seg in labels[fname] for sp in seg["species"].split(";")}
    sparse_files.append((fname, len(species_set)))

sparse_files.sort(key=lambda x: x[1])  # fewest species first
print("Sparse files with fewest species:")
for fname, n in sparse_files[:5]:
    print(f"  {fname}: {n} species")

In [ ]:
# Plot the sparse file (update filename after running cell above)
plot_separation(
    "BC2026_Train_0006_S09_20250828_000000.ogg",
    start_sec=0,
    duration_sec=15,
    save_name="sparse_quiet",
)

## Summary of channel activity across all 66 files

In [ ]:
# Compute RMS per channel across all 66 files
all_rms = []
for npz_path in sorted(SEPARATED_DIR.glob("*.npz")):
    data = np.load(npz_path)
    channels = data["channels"]
    rms = [np.sqrt(np.mean(channels[i] ** 2)) for i in range(4)]
    n_active = sum(1 for r in rms if r > 0.01)
    all_rms.append({"file": npz_path.stem, "rms": rms, "active": n_active})

# Distribution of active channels
active_counts = [r["active"] for r in all_rms]
print("Active channels (RMS > 0.01) distribution:")
for n in range(5):
    count = active_counts.count(n)
    print(f"  {n} active: {count} files ({count / len(all_rms) * 100:.0f}%)")

# Mean RMS per channel
mean_rms = np.mean([r["rms"] for r in all_rms], axis=0)
print(f"\nMean RMS per channel: {[f'{r:.4f}' for r in mean_rms]}")
ratio = mean_rms[0] / np.mean(mean_rms[1:])
print(f"Channel 0 dominates: {ratio:.1f}x higher RMS than channels 1-3")